<a href="https://colab.research.google.com/github/chk-33/large-language-modelloing/blob/main/lara%20finetunig.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

urllib.request.urlretrieve(
    url,
    "tinyshakespeare.txt"
)
with open(
    "tinyshakespeare.txt",
    "r",
    encoding="utf-8"
) as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {
    ch: i
    for i, ch in enumerate(chars)
}
itos = {
    i: ch
    for i, ch in enumerate(chars)
}
data = torch.tensor(
    [stoi[ch] for ch in text],
    dtype=torch.long
)
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

# Model configuration
vocab_size = 65
d_model = 128
num_layers = 2
num_heads = 2
kv_heads = 2
d_ff = 512
num_experts = 4
top_k = 2
max_seq_len = 256
dropout = 0.1

# Training configuration
batch_size = 32
block_size = 64
max_steps = 2000
learning_rate = 1e-3
min_learning_rate = 1e-4
warmup_steps = 100
weight_decay = 0.1
grad_clip = 1.0
eval_interval = 250
eval_iters = 20

# MoE configuration
capacity_factor = 1.25

# Auxiliary loss weight
aux_loss_weight = 0.01

# ============================================================
# SHAKESPEARE BATCH LOADER
# ============================================================

def get_batch(
    data,
    batch_size,
    block_size,
    device
):
    ix = torch.randint(
        0,
        len(data) - block_size - 1,
        (batch_size,)
    )
    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])
    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])
    return x.to(device), y.to(device)

class RMSNorm(nn.Module):
    def __init__(
        self,
        d_model,
        eps=1e-6
    ):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(
            torch.ones(d_model)
        )

    def forward(self, x):
        rms = torch.sqrt(
            torch.mean(
                x ** 2,
                dim=-1,
                keepdim=True
            )
            + self.eps
        )
        x = x / rms
        return self.weight * x

class RoPE(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len=2048,
        base=10000
    ):
        super().__init__()
        assert head_dim % 2 == 0
        self.head_dim = head_dim
        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )
        positions = torch.arange(
            max_seq_len
        ).float()
        freqs = torch.outer(
            positions,
            inv_freq
        )
        self.register_buffer(
            "cos",
            freqs.cos()
        )
        self.register_buffer(
            "sin",
            freqs.sin()
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        S = q.size(-2)
        positions = torch.arange(
            position_offset,
            position_offset + S,
            device=q.device
        )
        cos = self.cos[positions]
        sin = self.sin[positions]
        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)
        q1 = q[..., ::2]
        q2 = q[..., 1::2]
        k1 = k[..., ::2]
        k2 = k[..., 1::2]
        q_rotated = torch.stack(
            [
                q1 * cos - q2 * sin,
                q1 * sin + q2 * cos
            ],
            dim=-1
        )
        k_rotated = torch.stack(
            [
                k1 * cos - k2 * sin,
                k1 * sin + k2 * cos
            ],
            dim=-1
        )
        q_rotated = q_rotated.flatten(
            -2
        )
        k_rotated = k_rotated.flatten(
            -2
        )
        return (
            q_rotated,
            k_rotated
        )
class LoRALinear(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        r=8,
        alpha=16
    ):
        super().__init__()
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        self.weight = nn.Parameter(
            torch.empty(
                out_features,
                in_features
            )
        )
        self.weight.requires_grad = False
        self.A = nn.Parameter(
            torch.randn(
                r,
                in_features
            ) * 0.01
        )
        self.B = nn.Parameter(
            torch.zeros(
                out_features,
                r
            )
        )

    def forward(self, x):
        base = F.linear(
            x,
            self.weight
        )
        lora = F.linear(
            x,
            self.A
        )
        lora = F.linear(
            lora,
            self.B
        )
        lora = self.scaling * lora
        return base + lora
class GQA(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        kv_heads,
        max_seq_len=2048,
        dropout=0.0,
        lora_r=8,
        lora_alpha=16
    ):
        super().__init__()
        assert d_model % num_heads == 0
        assert num_heads % kv_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.kv_heads = kv_heads
        self.head_dim = (
            d_model // num_heads
        )
        self.groups = (
            num_heads // kv_heads
        )
        self.q_proj = LoRALinear(
            d_model,
            num_heads * self.head_dim,
            r=lora_r,
            alpha=lora_alpha
        )
        self.k_proj = nn.Linear(
            d_model,
            kv_heads * self.head_dim,
            bias=False
        )
        self.v_proj = LoRALinear(
            d_model,
            kv_heads * self.head_dim,
            r=lora_r,
            alpha=lora_alpha
        )
        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )
        self.rope = RoPE(
            self.head_dim,
            max_seq_len
        )
        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x, kv_cache=None, position_offset=0):
        B, S, D = x.shape
        q = self.q_proj(x)
        q = q.view(
            B,
            S,
            self.num_heads,
            self.head_dim
        )
        q = q.transpose(
            1,
            2
        )
        k = self.k_proj(x)
        k = k.view(
            B,
            S,
            self.kv_heads,
            self.head_dim
        )
        k = k.transpose(
            1,
            2
        )
        v = self.v_proj(x)
        v = v.view(
            B,
            S,
            self.kv_heads,
            self.head_dim
        )
        v = v.transpose(
            1,
            2
        )
        q, k = self.rope(
            q,
            k,
            position_offset=position_offset
        )

        if kv_cache is not None:
            k_prev, v_cache_prev = kv_cache
            k = torch.cat([k_prev, k], dim=-2)
            v = torch.cat([v_cache_prev, v], dim=-2)

        new_kv_cache = (k, v)

        k_rep = k.repeat_interleave(
            self.groups,
            dim=1
        )
        v_rep = v.repeat_interleave(
            self.groups,
            dim=1
        )
        scores = (
            q @ k_rep.transpose(
                -2,
                -1
            )
        )
        scores = (
            scores
            / math.sqrt(
                self.head_dim
            )
        )
        S_kv = k.size(-2)
        query_positions = (
             position_offset
               + torch.arange(
                S,
             device=x.device
                 )
                  )

        key_positions = torch.arange(
        S_kv,
        device=x.device
        )

        mask = (
        key_positions.unsqueeze(0)
        > query_positions.unsqueeze(1)
    )

        if kv_cache is None:
            out = F.scaled_dot_product_attention(
                q,
                k_rep,
                v_rep,
                dropout_p=(
                    self.dropout.p
                    if self.training
                    else 0.0
                ),
                is_causal=True
            )
        else:
            S_kv = k.size(-2)
            query_positions = (
                position_offset
                + torch.arange(
                    S,
                    device=x.device
                )
            )
            key_positions = torch.arange(
                S_kv,
                device=x.device
            )
            attention_mask = (
                key_positions.unsqueeze(0)
                <= query_positions.unsqueeze(1)
            )
            out = F.scaled_dot_product_attention(
                q,
                k_rep,
                v_rep,
                attn_mask=attention_mask,
                dropout_p=(
                    self.dropout.p
                    if self.training
                    else 0.0
                ),
                is_causal=False
            )
        out = out.transpose(
            1,
            2
        ).contiguous()
        out = out.view(
            B,
            S,
            D
        )
        out = self.out_proj(
            out
        )
        return out, new_kv_cache

class Expert(nn.Module):
    def __init__(
        self,
        d_model,
        d_ff
    ):
        super().__init__()
        self.gate_proj = nn.Linear(
             d_model,
            d_ff,
            bias=False
        )
        self.value_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False
        )
        self.output_proj = nn.Linear(
            d_ff,
            d_model,
            bias=False
        )

    def forward(self, x):
        gate_pre = self.gate_proj(x)
        value = self.value_proj(x)
        gate = F.silu(gate_pre)
        hidden = gate * value
        output = self.output_proj(hidden)
        return output

# ============================================================
# LOAD BALANCING LOSS
# ============================================================

def load_balancing_loss(
    router_probs,
    top_indices,
    num_experts
):
    P = router_probs.mean(
        dim=(0, 1)
    )
    assignment_mask = F.one_hot(
        top_indices,
        num_classes=num_experts
    ).float()
    f = assignment_mask.mean(
        dim=(0, 1, 2)
    )
    loss = (
        num_experts
        * torch.sum(f * P)
    )
    return loss

# ============================================================
# MoE DISPATCHER
# ============================================================

class MoEDispatcher:
    def __init__(
        self,
        num_experts,
        top_k,
        capacity
    ):
        self.num_experts = num_experts
        self.top_k = top_k
        self.capacity = capacity

    def dispatch(
        self,
        x,
        top_indices
    ):
        B, S, D = x.shape
        T = B * S
        x_flat = x.reshape(T, D)
        expert_inputs = []
        token_indices_per_expert = []
        k_indices_per_expert = []
        dropped_assignments = 0

        for expert_id in range(
            self.num_experts
        ):
            mask = (top_indices == expert_id)
            mask = mask.reshape(T, self.top_k)
            token_indices, k_indices = torch.where(mask)
            num_assignments = token_indices.numel()

            if num_assignments > self.capacity:
                dropped_assignments += (
                    num_assignments
                    - self.capacity
                )
                token_indices = token_indices[:self.capacity]
                k_indices = k_indices[:self.capacity]

            expert_input = x_flat[token_indices]
            expert_inputs.append(expert_input)
            token_indices_per_expert.append(token_indices)
            k_indices_per_expert.append(k_indices)

        return (
            expert_inputs,
            token_indices_per_expert,
            k_indices_per_expert,
            dropped_assignments
        )

# ============================================================
# Mixture of Experts
# ============================================================

class MoE(nn.Module):
    def __init__(
        self,
        d_model,
        d_ff,
        num_experts=4,
        top_k=2,
        capacity_factor=1.25
    ):
        super().__init__()
        assert num_experts >= 1
        assert 1 <= top_k <= num_experts
        assert capacity_factor > 0
        self.d_model = d_model
        self.d_ff = d_ff
        self.num_experts = num_experts
        self.top_k = top_k
        self.capacity_factor = capacity_factor

        self.router = nn.Linear(
            d_model,
            num_experts,
            bias=False
        )
        self.experts = nn.ModuleList(
            [
                Expert(d_model, d_ff)
                for _ in range(num_experts)
            ]
        )

    def forward(self, x):
        B, S, D = x.shape
        assert D == self.d_model
        T = B * S

        router_logits = self.router(x)
        router_probs = F.softmax(router_logits, dim=-1)
        top_weights, top_indices = torch.topk(
            router_probs,
            k=self.top_k,
            dim=-1
        )
        top_weights = (
            top_weights
            / top_weights.sum(
                dim=-1,
                keepdim=True
            ).clamp_min(1e-9)
        )
        capacity = math.ceil(
            (
                T * self.top_k
                / self.num_experts
            )
            * self.capacity_factor
        )
        dispatcher = MoEDispatcher(
            num_experts=self.num_experts,
            top_k=self.top_k,
            capacity=capacity
        )
        (
            expert_inputs,
            token_indices_per_expert,
            k_indices_per_expert,
            dropped_assignments
        ) = dispatcher.dispatch(x, top_indices)

        expert_outputs = []
        for expert_id in range(self.num_experts):
            expert_input = expert_inputs[expert_id]
            if expert_input.numel() == 0:
                expert_output = expert_input
            else:
                expert_output = self.experts[expert_id](expert_input)
            expert_outputs.append(expert_output)

        x_flat = x.reshape(T, D)
        output = torch.zeros_like(x_flat)
        top_weights_flat = top_weights.reshape(T, self.top_k)

        for expert_id in range(self.num_experts):
            token_indices = token_indices_per_expert[expert_id]
            k_indices = k_indices_per_expert[expert_id]
            if token_indices.numel() == 0:
                continue
            expert_output = expert_outputs[expert_id]
            weights = top_weights_flat[token_indices, k_indices]
            weights = weights.unsqueeze(-1)
            weighted_output = expert_output * weights
            output.index_add_(0, token_indices, weighted_output)

        output = output.view(B, S, D)
        return (
            output,
            router_logits,
            router_probs,
            top_indices,
            dropped_assignments
        )

class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model=128,
        num_heads=4,
        kv_heads=2,
        d_ff=512,
        num_experts=4,
        top_k=2,
        max_seq_len=2048,
        dropout=0.0,
        lora_r=8,
        lora_alpha=16
    ):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = GQA(
            d_model=d_model,
            num_heads=num_heads,
            kv_heads=kv_heads,
            max_seq_len=max_seq_len,
            dropout=dropout,
            lora_r=lora_r,
            lora_alpha=lora_alpha
        )
        self.norm2 = RMSNorm(d_model)
        self.moe = MoE(
            d_model=d_model,
            d_ff=d_ff,
            num_experts=num_experts,
            top_k=top_k,
            capacity_factor=capacity_factor
        )

    def forward(self, x, kv_cache=None, position_offset=0):
        residual = x
        x = self.norm1(x)
        x, new_kv_cache = self.attention(x, kv_cache=kv_cache, position_offset=position_offset)
        x = residual + x

        residual = x
        x = self.norm2(x)
        (
            moe_output,
            router_logits,
            router_probs,
            top_indices,
            dropped_tokens
        ) = self.moe(x)
        x = residual + moe_output

        aux_loss = load_balancing_loss(
            router_probs,
            top_indices,
            self.moe.num_experts
        )
        return (
            x,
            new_kv_cache,
            aux_loss,
            router_logits,
            router_probs,
            top_indices,
            dropped_tokens
        )

# ============================================================
# COMPLETE TRANSFORMER MODEL
# ============================================================

class TransformerModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        kv_heads,
        d_ff,
        num_experts,
        top_k,
        max_seq_len,
        dropout=0.1,
        lora_r=8,
        lora_alpha=16
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    d_model=d_model,
                    num_heads=num_heads,
                    kv_heads=kv_heads,
                    d_ff=d_ff,
                    num_experts=num_experts,
                    top_k=top_k,
                    max_seq_len=max_seq_len,
                    dropout=dropout,
                    lora_r=lora_r,
                    lora_alpha=lora_alpha
                )
                for _ in range(num_layers)
            ]
        )
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids, targets=None, kv_caches=None, position_offset=0):
        x = self.embedding(input_ids)
        total_aux_loss = torch.zeros((), device=x.device)
        total_dropped_tokens = 0
        router_information = []
        new_kv_caches = []

        for i, block in enumerate(self.blocks):
            kv_cache = kv_caches[i] if kv_caches is not None else None
            (
                x,
                new_kv_cache,
                aux_loss,
                router_logits,
                router_probs,
                top_indices,
                dropped_tokens
            ) = block(x, kv_cache=kv_cache, position_offset=position_offset)
            new_kv_caches.append(new_kv_cache)
            total_aux_loss = total_aux_loss + aux_loss
            total_dropped_tokens += dropped_tokens
            router_information.append({
                "logits": router_logits,
                "probs": router_probs,
                "top_indices": top_indices
            })

        x = self.final_norm(x)
        logits = self.lm_head(x)
        lm_loss = None

        if targets is not None:
            lm_loss = F.cross_entropy(
                logits.reshape(-1, self.vocab_size),
                targets.reshape(-1)
            )

        loss = None
        if lm_loss is not None:
            loss = (
                lm_loss
                + aux_loss_weight
                * total_aux_loss
            )

        return (
            logits,
            loss,
            lm_loss,
            total_aux_loss,
            total_dropped_tokens,
            router_information,
            new_kv_caches
        )

# ============================================================
# CREATE MODEL & CHECKPOINT RECOVERY
# ============================================================
lora_r = 8
lora_alpha = 16

model = TransformerModel(
    vocab_size=vocab_size,
    d_model=d_model,
    num_layers=num_layers,
    num_heads=num_heads,
    kv_heads=kv_heads,
    d_ff=d_ff,
    num_experts=num_experts,
    top_k=top_k,
    max_seq_len=max_seq_len,
    dropout=dropout,
    lora_r=lora_r,
    lora_alpha=lora_alpha
).to(device)

try:
    checkpoint = torch.load("best_model.pt", map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print("Successfully loaded best model weights from checkpoint.")
except Exception as e:
    print("Could not find or load checkpoint; proceeding with initialized model. Details:", e)

# Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Enable training only for LoRA parameters
for module in model.modules():
    if isinstance(module, LoRALinear):
        module.A.requires_grad = True
        module.B.requires_grad = True

# ============================================================
@torch.no_grad()
def evaluate_model(
    model,
    data,
    batch_size,
    block_size,
    eval_iters,
    device
):
    model.eval()

    losses = []

    for _ in range(eval_iters):

        x, y = get_batch(
            data,
            batch_size,
            block_size,
            device
        )

        (
            logits,
            loss,
            lm_loss,
            aux_loss,
            dropped_tokens,
            router_information,
            kv_caches
        ) = model(
            x,
            targets=y
        )

        losses.append(
            lm_loss.item()
        )

    model.train()

    mean_loss = sum(losses) / len(losses)

    perplexity = math.exp(mean_loss)

    return mean_loss, perplexity
# LEARNING RATE SCHEDULE
# ============================================================

def get_lr(step, max_steps, lr_max, lr_min, warmup_steps):
    if step < warmup_steps:
        return lr_max * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return lr_min + cosine * (lr_max - lr_min)

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(model, data, batch_size, block_size, device, eval_iters):
    model.eval()
    total_losses = []
    for _ in range(eval_iters):
        input_ids, targets = get_batch(data, batch_size, block_size, device)
        _, loss, _, _, _, _, _ = model(input_ids, targets)
        total_losses.append(loss.item())
    model.train()
    return sum(total_losses) / len(total_losses)

# ============================================================
# TRAINING PIPELINE
# ============================================================

print("Training the model to minimize errors...")
trainable_params = [
    p
    for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=1e-4,
    weight_decay=0.01
)
for step in range(max_steps):
    lr = get_lr(step, max_steps, learning_rate, min_learning_rate, warmup_steps)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    input_ids, targets = get_batch(train_data, batch_size, block_size, device)
    logits, loss, lm_loss, _, _, _, _ = model(input_ids, targets)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if step % eval_interval == 0 or step == max_steps - 1:
        val_loss = evaluate(model, val_data, batch_size, block_size, device, eval_iters)
        print(f"Step {step:4d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f}")

# Save trained state
torch.save({"model_state_dict": model.state_dict()}, "best_model.pt")

# ============================================================
# RE-ENGINEERED GENERATION WITH KV CACHING & EXPONENTIAL REPETITION PENALTY
# ============================================================

@torch.no_grad()
def generate(
    model,
    input_ids,
    max_new_tokens,
    max_seq_len,
    temperature=1.0,
    top_p=0.92,
    top_k=10,
    repetition_penalty=1.5
):
    model.eval()
    kv_caches = None

    # Process the initial prompt to populate the KV cache
    logits, _, _, _, _, _, kv_caches = model(input_ids, kv_caches=kv_caches, position_offset=0)
    logits = logits[:, -1, :]

    # Apply repetition penalty to initial logits if applicable
    if repetition_penalty != 1.0:
        for i in range(input_ids.size(0)):
            unique_tokens, counts = torch.unique(input_ids[i], return_counts=True)
            for token, count in zip(unique_tokens, counts):
                token_id = token.item()
                logits[i, token_id] -= float(count) * repetition_penalty

    logits = logits / temperature
    if top_k is not None:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float('Inf')

    if top_p is not None:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0
        for i in range(logits.size(0)):
            indices_to_remove = sorted_indices[i][sorted_indices_to_remove[i]]
            logits[i, indices_to_remove] = float("-inf")

    next_token = torch.argmax(logits, dim=-1, keepdim=True)
    input_ids = torch.cat((input_ids, next_token), dim=1)

    for step in range(1, max_new_tokens):
        # For generation, we only pass the *last* token
        curr_input = input_ids[:, [-1]]
        position_offset = input_ids.size(1) - 2

        logits, _, _, _, _, _, kv_caches = model(
            curr_input,
            kv_caches=kv_caches,
            position_offset=position_offset
        )
        logits = logits[:, -1, :]

        if repetition_penalty != 1.0:
            for i in range(input_ids.size(0)):
                unique_tokens, counts = torch.unique(input_ids[i], return_counts=True)
                for token, count in zip(unique_tokens, counts):
                    token_id = token.item()
                    logits[i, token_id] -= float(count) * repetition_penalty

        logits = logits / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')

        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            for i in range(logits.size(0)):
                indices_to_remove = sorted_indices[i][sorted_indices_to_remove[i]]
                logits[i, indices_to_remove] = float("-inf")

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.argmax(
            logits,
            dim=-1,
            keepdim=True,
        )
        input_ids = torch.cat((input_ids, next_token), dim=1)
    return input_ids

# ============================================================
# EXAMPLE INFERENCE USAGE
# ============================================================
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
prompt_text = "First Citizen:"
prompt_tokens = torch.tensor([encode(prompt_text)], dtype=torch.long, device=device)

MAX_NEW_TOKENS = 120
TEMPERATURE = 0.85
TOP_K = 35
TOP_P=0.90

generated_indices = generate(
    model=model,
    input_ids=prompt_tokens,
    max_new_tokens=MAX_NEW_TOKENS,
    max_seq_len=max_seq_len,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=1.5
)
generated_text = decode(generated_indices[0].tolist())
print("================ GENERATED OUTPUT =================")
print(generated_text)
val_loss, val_perplexity = evaluate_model(
    model,
    val_data,
    batch_size,
    block_size,
    eval_iters,
    device
)

print("Validation Loss:", val_loss)
print("Validation Perplexity:", val_perplexity)
@torch.no_grad()
def evaluate_expert_usage(
    model,
    data,
    batch_size,
    block_size,
    eval_iters,
    device
):
    model.eval()

    num_experts = model.blocks[0].moe.num_experts

    expert_counts = torch.zeros(
        num_experts,
        device=device
    )

    total_assignments = 0

    total_dropped = 0

    for _ in range(eval_iters):

        x, y = get_batch(
            data,
            batch_size,
            block_size,
            device
        )

        (
            logits,
            loss,
            lm_loss,
            aux_loss,
            dropped_tokens,
            router_information,
            kv_caches
        ) = model(x)

        for info in router_information:

            top_indices = info["top_indices"]

            for expert_id in range(num_experts):

                expert_counts[expert_id] += (
                    (top_indices == expert_id)
                    .sum()
                )

            total_assignments += (
                top_indices.numel()
            )

        total_dropped += dropped_tokens

    usage_percent = (
        expert_counts
        / total_assignments
        * 100
    )

    drop_rate = (
        total_dropped
        / total_assignments
        * 100
    )

    model.train()

    return usage_percent, drop_rate
expert_usage, drop_rate = evaluate_expert_usage(
    model,
    val_data,
    batch_size,
    block_size,
    eval_iters,
    device
)

print("\n===== MOE EVALUATION =====")

for i, usage in enumerate(expert_usage):
    print(
        f"Expert {i}: {usage.item():.2f}%"
    )

print(
    f"Dropped Assignment Rate: "
    f"{drop_rate:.2f}%"
)
trainable = 0
total = 0

for name, param in model.named_parameters():

    total += param.numel()

    if param.requires_grad:
        trainable += param.numel()
        print(
            name,
            param.shape
        )

print("\nTotal:", total)
print("Trainable:", trainable)
print(
    "Trainable percentage:",
    100 * trainable / total
)

Successfully loaded best model weights from checkpoint.
Training the model to minimize errors...
Step    0 | Train Loss: 2.0682 | Val Loss: 2.0907
Step  250 | Train Loss: 2.0963 | Val Loss: 2.0843
Step  500 | Train Loss: 2.0580 | Val Loss: 2.1023
Step  750 | Train Loss: 2.0597 | Val Loss: 2.0974
Step 1000 | Train Loss: 2.1034 | Val Loss: 2.0854
Step 1250 | Train Loss: 2.0697 | Val Loss: 2.0943
Step 1500 | Train Loss: 2.0197 | Val Loss: 2.0922
Step 1750 | Train Loss: 2.0246 | Val Loss: 2.0992
Step 1999 | Train Loss: 2.0820 | Val Loss: 2.0835
================ GENERATED OUTPUT =================
First Citizen:
Though all, my beaved,
Whought's;
Andwark.
Toncefford!--ply?
ORDYOLINGESSack:
My, with'spubjup.'THABUKENRDYYY:
WIZZAPJU
Validation Loss: 2.084411770105362
Validation Perplexity: 8.039860803997435

===== MOE EVALUATION =====
Expert 0: 24.58%
Expert 1: 22.11%
Expert 2: 28.71%
Expert 3: 24.61%
Dropped Assignment Rate: 0.01%
blocks.0.attention.q_proj.A torch.Size([8, 128])
blocks.0.atten

In [8]:
try:
    checkpoint = torch.load("best_model.pt", map_location=device)

    # Load old checkpoint while allowing the newly added A/B parameters
    missing_keys, unexpected_keys = model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=False
    )

    print("Successfully loaded best model weights from checkpoint.")

    print("\nMissing keys:")
    for key in missing_keys:
        print(" ", key)

    print("\nUnexpected keys:")
    for key in unexpected_keys:
        print(" ", key)

except Exception as e:
    print(
        "Could not find or load checkpoint; "
        "proceeding with initialized model. Details:",
        e
    )



Successfully loaded best model weights from checkpoint.

Missing keys:

Unexpected keys:


In [10]:
checkpoint = torch.load("best_model.pt", map_location=device)

state_dict = checkpoint["model_state_dict"]

print("Number of checkpoint parameters:", len(state_dict))

for key in state_dict:
    if "q_proj.A" in key or "q_proj.B" in key or \
       "v_proj.A" in key or "v_proj.B" in key:
        print(key, state_dict[key].shape)
# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only Q/V low-rank adapters
for name, param in model.named_parameters():
    if (
        "q_proj.A" in name or
        "q_proj.B" in name or
        "v_proj.A" in name or
        "v_proj.B" in name
    ):
        param.requires_grad = True

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print(
    "Trainable percentage:",
    100 * trainable_params / total_params
)

print("\nTrainable parameters:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {tuple(param.shape)}")

Number of checkpoint parameters: 53
blocks.0.attention.q_proj.A torch.Size([8, 128])
blocks.0.attention.q_proj.B torch.Size([128, 8])
blocks.0.attention.v_proj.A torch.Size([8, 128])
blocks.0.attention.v_proj.B torch.Size([128, 8])
blocks.1.attention.q_proj.A torch.Size([8, 128])
blocks.1.attention.q_proj.B torch.Size([128, 8])
blocks.1.attention.v_proj.A torch.Size([8, 128])
blocks.1.attention.v_proj.B torch.Size([128, 8])
Total parameters: 1722112
Trainable parameters: 8192
Trainable percentage: 0.47569496060651106

Trainable parameters:
blocks.0.attention.q_proj.A: (8, 128)
blocks.0.attention.q_proj.B: (128, 8)
blocks.0.attention.v_proj.A: (8, 128)
blocks.0.attention.v_proj.B: (128, 8)
blocks.1.attention.q_proj.A: (8, 128)
blocks.1.attention.q_proj.B: (128, 8)
blocks.1.attention.v_proj.A: (8, 128)
blocks.1.attention.v_proj.B: (128, 8)
